In [11]:
import os
import cv2
import numpy as np
import tifffile as tiff
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import distance
from scipy.optimize import linear_sum_assignment
from skimage.measure import regionprops

# ======================= User Inputs =======================
MASK_A_PATH = r"C:\Users\oxfil\0916_cellpose_formathcedimages\10X\Position_similar_ROIA004_10_new_seg.npy" #Mask file for 10X upscaled Capacitance
MASK_B_PATH = r"C:\Users\oxfil\Cell_optical_media_comparision\July_22_ROI_A_004_RGB_seg.npy" # Mask file for Optical images
IMG_B_PATH  = r"C:\Users\oxfil\Cell_optical_media_comparision\July_22_ROI_A_004_RGB.tif" # Images for overlay 
OUT_DIR = r"."

# Scale(px/um)
PX_PER_UM_A = 1.00
PX_PER_UM_B = 2.68  # scale = PX_PER_UM_B / PX_PER_UM_A

# Matching Window
COARSE_RADIUS_YX = (500, 500)
COARSE_STRIDE    = 60
REFINE_RADIUS_YX = (60, 60)
REFINE_STRIDE    = 2

# Matching parameter
FINAL_MAX_DIST_BPX   = 100.0
UNMATCHED_PENALTY_B  = 200.0

UPSCALE_VIS = 1
INVERT_VIEW = False # some images with inverted value
# ==========================================================


def load_mask(path):
    ext = os.path.splitext(path.lower())[1]
    if ext in (".tif", ".tiff"):
        return np.asarray(tiff.imread(path)).astype(np.int32)
    elif ext == ".npy":
        data = np.load(path, allow_pickle=True)
        if isinstance(data, dict) and "masks" in data:
            return np.asarray(data["masks"]).astype(np.int32)
        if isinstance(data, np.ndarray):
            if data.dtype == object and isinstance(data.flat[0], dict) and "masks" in data.flat[0]:
                return np.asarray(data.flat[0]["masks"]).astype(np.int32)
            if data.ndim >= 2:
                return data.astype(np.int32)
        raise TypeError("Unsupported npy structure.")
    else:
        arr = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if arr is None:
            raise FileNotFoundError(path)
        return np.asarray(arr).astype(np.int32)


def nearest_resample_labels(src, scale):
    H, W = src.shape
    W2 = int(round(W * scale))
    H2 = int(round(H * scale))
    return cv2.resize(src, (W2, H2), interpolation=cv2.INTER_NEAREST)


def to_binary(mask):
    return (mask > 0).astype(np.uint8)


def overlap_count(binA, binB, dy, dx):
    Ha, Wa = binA.shape
    Hb, Wb = binB.shape
    dy = int(dy)
    dx = int(dx)

    y0A = max(0, -dy)
    y1A = min(Ha, Hb - dy)
    x0A = max(0, -dx)
    x1A = min(Wa, Wb - dx)

    if y1A <= y0A or x1A <= x0A:
        return 0

    y0B = max(0, dy)
    y1B = y0B + (y1A - y0A)
    x0B = max(0, dx)
    x1B = x0B + (x1A - x0A)

    Aov = binA[y0A:y1A, x0A:x1A]
    Bov = binB[y0B:y1B, x0B:x1B]
    return int((Aov * Bov).sum())


def grid_search_peak(binA, binB, center_dy, center_dx, rad_yx, stride, tag):
    ry, rx = rad_yx
    best = {"ov": -1, "dy": 0, "dx": 0}
    ys = range(center_dy - ry, center_dy + ry + 1, stride)
    xs = range(center_dx - rx, center_dx + rx + 1, stride)
    tried = 0

    for dy in ys:
        for dx in xs:
            tried += 1
            ov = overlap_count(binA, binB, dy, dx)
            if ov > best["ov"]:
                best.update({"ov": ov, "dy": int(dy), "dx": int(dx)})

    print(f"[{tag}] tried={tried}  best_ov={best['ov']}  shift=({best['dy']},{best['dx']})")
    return best


def translate_labels_to_canvas(src, dy, dx, canvas_shape):
    HB, WB = canvas_shape
    out = np.zeros((HB, WB), dtype=src.dtype)
    dy = int(dy)
    dx = int(dx)
    H, W = src.shape

    y0s = max(0, -dy)
    y1s = min(W if False else H, HB - dy)
    x0s = max(0, -dx)
    x1s = min(W, WB - dx)

    y0d = max(0, dy)
    y1d = y0d + (y1s - y0s)
    x0d = max(0, dx)
    x1d = x0d + (x1s - x0s)

    if y1s > y0s and x1s > x0s:
        out[y0d:y1d, x0d:x1d] = src[y0s:y1s, x0s:x1s]
    return out


def extract_features(mask):
    """
    For Shape-only PCA / pair similarity
    """
    rows = []
    for p in regionprops(mask):
        y, x = p.centroid
        area = float(p.area)
        peri = float(p.perimeter)

        major = float(p.major_axis_length)
        minor = float(p.minor_axis_length)
        aspect = (major / minor) if minor > 0 else np.nan
        circ = (4.0 * np.pi * area) / (peri**2 + 1e-12)

        rows.append(dict(
            label=int(p.label),
            cx=float(x),
            cy=float(y),

            # size-related
            area=area,
            perimeter=peri,
            major_axis_length=major,
            minor_axis_length=minor,
            equivalent_diameter=float(p.equivalent_diameter),

            aspect=aspect,
            circularity=circ,
            solidity=float(p.solidity) if hasattr(p, "solidity") else np.nan,
            eccentricity=float(p.eccentricity) if hasattr(p, "eccentricity") else np.nan,
            extent=float(p.extent) if hasattr(p, "extent") else np.nan,
            
            orientation=float(p.orientation) if hasattr(p, "orientation") else np.nan,
            convex_area=float(p.convex_area) if hasattr(p, "convex_area") else np.nan,
            bbox_area=float((p.bbox[2] - p.bbox[0]) * (p.bbox[3] - p.bbox[1])),
        ))
    return pd.DataFrame(rows)


def hungarian_with_unmatched(Axy, Bxy, max_dist, unmatched_penalty):
    Axy = np.asarray(Axy, float)
    Bxy = np.asarray(Bxy, float)
    nA, nB = len(Axy), len(Bxy)

    if nA == 0 and nB == 0:
        return [], [], []
    if nA == 0:
        return [], [], list(range(nB))
    if nB == 0:
        return [], list(range(nA)), []

    C = distance.cdist(Axy, Bxy)
    BIG = 1e9
    C = np.where(C <= max_dist, C, BIG)

    PA = np.full((nA, nA), unmatched_penalty, float)
    PB = np.full((nB, nB), unmatched_penalty, float)
    cost = np.vstack([np.hstack([C, PA]), np.hstack([PB, np.zeros((nB, nA))])])

    r, c = linear_sum_assignment(cost)
    matches = []
    usedA = np.zeros(nA, bool)
    usedB = np.zeros(nB, bool)

    for i, j in zip(r, c):
        if i < nA and j < nB and C[i, j] < BIG:
            matches.append((i, j, float(C[i, j])))
            usedA[i] = True
            usedB[j] = True

    unA = [i for i in range(nA) if not usedA[i]]
    unB = [j for j in range(nB) if not usedB[j]]
    return matches, unA, unB


def load_bg_image(path, shape_hw, invert=False):
    H, W = shape_hw
    if path and os.path.exists(path):
        img = tiff.imread(path) if path.lower().endswith((".tif", ".tiff")) else cv2.imread(path, cv2.IMREAD_UNCHANGED)

        if img.ndim == 2:
            img8 = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            if invert:
                img8 = 255 - img8
            bg = cv2.cvtColor(img8, cv2.COLOR_GRAY2BGR)
        else:
            img8 = img if img.dtype == np.uint8 else cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            if invert:
                img8 = 255 - img8
            bg = img8[:, :, :3]

        return cv2.resize(bg, (W, H), interpolation=cv2.INTER_LANCZOS4)
    else:
        return np.zeros((H, W, 3), dtype=np.uint8)


def draw_vectors(bg_bgr, Axy_B, Bxy_B, matches, upscale, out_png):
    H, W = bg_bgr.shape[:2]
    canvas = cv2.resize(bg_bgr, (W * upscale, H * upscale), interpolation=cv2.INTER_LANCZOS4)

    def T(pt):
        return (int(round(pt[0] * upscale)), int(round(pt[1] * upscale)))

    r = max(2, upscale // 2)

    for x, y in Bxy_B:
        cv2.circle(canvas, T((x, y)), r, (255, 200, 0), -1)
    for x, y in Axy_B:
        cv2.circle(canvas, T((x, y)), r, (0, 220, 0), -1)

    for iA, iB, _ in matches:
        a = Axy_B[iA]
        b = Bxy_B[iB]
        cv2.arrowedLine(
            canvas, T((a[0], a[1])), T((b[0], b[1])),
            (255, 255, 255), max(1, upscale), tipLength=0.25, line_type=cv2.LINE_AA
        )

    cv2.imwrite(out_png, canvas)


def compute_iou_for_pair(A_on_B, Bmask, labelA, labelB):
    Abin = (A_on_B == int(labelA))
    Bbin = (Bmask == int(labelB))
    inter = int(np.logical_and(Abin, Bbin).sum())
    uni = int(np.logical_or(Abin, Bbin).sum())
    return (inter / uni) if uni > 0 else 0.0


def scatter_and_corr(x, y, name, out_dir):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    if len(x) < 2:
        print(f"[{name}] no effective match")
        return

    r = np.corrcoef(x, y)[0, 1]
    a, b = np.polyfit(x, y, 1)
    xx = np.linspace(x.min(), x.max(), 100)
    yy = a * xx + b

    plt.figure(figsize=(5, 5))
    plt.scatter(x, y, s=12, alpha=0.6)
    plt.plot(xx, yy, lw=2)
    plt.xlabel(f"{name} (A)")
    plt.ylabel(f"{name} (B)")
    plt.title(f"{name}: r={r:.3f}")
    plt.tight_layout()

    out = os.path.join(out_dir, f"scatter_{name}.png")
    plt.savefig(out, dpi=200)
    plt.close()
    print(f"[Saved] {out}")


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    # --- Load ---
    A = load_mask(MASK_A_PATH)
    B = load_mask(MASK_B_PATH)
    print(f"[SHAPE] A={A.shape}, B={B.shape}")

    # --- Scale to B for alignment ---
    scale = PX_PER_UM_B / PX_PER_UM_A
    inv_scale = 1.0 / scale
    A_scaled = nearest_resample_labels(A, scale)
    print(f"[SCALE] {scale:.4f}  A_scaled={A_scaled.shape}  B={B.shape}")

    # --- AND-overlap peak (coarse→refine) ---
    binA = to_binary(A_scaled)
    binB = to_binary(B)

    best_c = grid_search_peak(binA, binB, 0, 0, COARSE_RADIUS_YX, COARSE_STRIDE, "COARSE")
    best_r = grid_search_peak(binA, binB, best_c["dy"], best_c["dx"], REFINE_RADIUS_YX, REFINE_STRIDE, "REFINE")
    dy_Bpx, dx_Bpx, ov = best_r["dy"], best_r["dx"], best_r["ov"]
    print(f"[SELECT] dy_B={dy_Bpx}, dx_B={dx_Bpx}, overlap_px={ov}")

    # --- Place A on B-canvas (labels preserved) ---
    A_on_B = translate_labels_to_canvas(A_scaled, dy_Bpx, dx_Bpx, canvas_shape=B.shape)
    tiff.imwrite(os.path.join(OUT_DIR, "A_scaled_shifted_on_B.tif"), A_on_B.astype(np.int32))

    # --- Features in ORIGINAL scale (A,B) ---
    dfA = extract_features(A)
    dfB = extract_features(B)

    # --- Build centroids in both spaces ---
    Axy_A = dfA[["cx", "cy"]].to_numpy()
    Bxy_B = dfB[["cx", "cy"]].to_numpy()
    Bxy_A = Bxy_B * inv_scale
    Axy_B = Axy_A * scale + np.array([dx_Bpx, dy_Bpx])[None, :]

    # --- Translation vector in ALL units ---
    dy_Apx = dy_Bpx * inv_scale
    dx_Apx = dx_Bpx * inv_scale
    dy_um = dy_Bpx / PX_PER_UM_B
    dx_um = dx_Bpx / PX_PER_UM_B

    # --- Matching in ORIGINAL A-scale ---
    FINAL_MAX_DIST_A = FINAL_MAX_DIST_BPX * inv_scale
    UNMATCHED_PENALTY_A = UNMATCHED_PENALTY_B * inv_scale
    Axy_A_shifted = Axy_A + np.array([dx_Apx, dy_Apx])[None, :]

    matches, unA, unB = hungarian_with_unmatched(
        Axy_A_shifted, Bxy_A, FINAL_MAX_DIST_A, UNMATCHED_PENALTY_A
    )
    print(f"[MATCH] matches={len(matches)}  unA={len(unA)}  unB={len(unB)}  gate_Apx={FINAL_MAX_DIST_A:.1f}")

    # --- Export CSVs ---
    pairs_rows = []
    for iA, iB, dist_Apx in matches:
        labelA = int(dfA.iloc[iA]["label"])
        labelB = int(dfB.iloc[iB]["label"])

        Ax_A, Ay_A = float(dfA.iloc[iA]["cx"]), float(dfA.iloc[iA]["cy"])
        Bx_B, By_B = float(dfB.iloc[iB]["cx"]), float(dfB.iloc[iB]["cy"])

        Ax_B = Ax_A * scale + dx_Bpx
        Ay_B = Ay_A * scale + dy_Bpx
        Bx_A = Bx_B * inv_scale
        By_A = By_B * inv_scale

        dist_Bpx = np.hypot((Ax_B - Bx_B), (Ay_B - By_B))
        dist_um = dist_Bpx / PX_PER_UM_B
        iou = compute_iou_for_pair(A_on_B, B, labelA, labelB)

        pairs_rows.append(dict(
            # indices / labels
            idxA=iA, idxB=iB, labelA=labelA, labelB=labelB,

            # translation
            shift_dx_Bpx=dx_Bpx, shift_dy_Bpx=dy_Bpx,
            shift_dx_Apx=dx_Apx, shift_dy_Apx=dy_Apx,
            shift_dx_um=dx_um, shift_dy_um=dy_um,

            # coordinates (both scales)
            Ax_Apx=Ax_A, Ay_Apx=Ay_A,
            Ax_Bpx=Ax_B, Ay_Bpx=Ay_B,
            Bx_Bpx=Bx_B, By_Bpx=By_B,
            Bx_Apx=Bx_A, By_Apx=By_A,

            # distances
            dist_Apx=float(dist_Apx),
            dist_Bpx=float(dist_Bpx),
            dist_um=float(dist_um),

            # original features
            areaA=dfA.iloc[iA]["area"],
            areaB=dfB.iloc[iB]["area"],
            periA=dfA.iloc[iA]["perimeter"],
            periB=dfB.iloc[iB]["perimeter"],

            majorA=dfA.iloc[iA]["major_axis_length"],
            majorB=dfB.iloc[iB]["major_axis_length"],
            minorA=dfA.iloc[iA]["minor_axis_length"],
            minorB=dfB.iloc[iB]["minor_axis_length"],
            equiv_diamA=dfA.iloc[iA]["equivalent_diameter"],
            equiv_diamB=dfB.iloc[iB]["equivalent_diameter"],

            aspectA=dfA.iloc[iA]["aspect"],
            aspectB=dfB.iloc[iB]["aspect"],
            circA=dfA.iloc[iA]["circularity"],
            circB=dfB.iloc[iB]["circularity"],
            solidityA=dfA.iloc[iA]["solidity"],
            solidityB=dfB.iloc[iB]["solidity"],
            eccentricityA=dfA.iloc[iA]["eccentricity"],
            eccentricityB=dfB.iloc[iB]["eccentricity"],
            extentA=dfA.iloc[iA]["extent"],
            extentB=dfB.iloc[iB]["extent"],
            orientationA=dfA.iloc[iA]["orientation"],
            orientationB=dfB.iloc[iB]["orientation"],
            convex_areaA=dfA.iloc[iA]["convex_area"],
            convex_areaB=dfB.iloc[iB]["convex_area"],
            bbox_areaA=dfA.iloc[iA]["bbox_area"],
            bbox_areaB=dfB.iloc[iB]["bbox_area"],

            # physical area
            areaA_um2=dfA.iloc[iA]["area"] / (PX_PER_UM_A**2),
            areaB_um2=dfB.iloc[iB]["area"] / (PX_PER_UM_B**2),

            # quality
            iou_Bscale=float(iou)
        ))

    df_pairs = pd.DataFrame(pairs_rows)
    df_pairs.to_csv(os.path.join(OUT_DIR, "pairs.csv"), index=False)
    print("[Saved] pairs.csv")

    if len(unA) > 0:
        pd.DataFrame({
            "idxA": unA,
            "labelA": [int(dfA.iloc[i]["label"]) for i in unA]
        }).to_csv(os.path.join(OUT_DIR, "unmatched_A.csv"), index=False)

    if len(unB) > 0:
        pd.DataFrame({
            "idxB": unB,
            "labelB": [int(dfB.iloc[j]["label"]) for j in unB]
        }).to_csv(os.path.join(OUT_DIR, "unmatched_B.csv"), index=False)

    pd.DataFrame([dict(
        PX_PER_UM_A=PX_PER_UM_A, PX_PER_UM_B=PX_PER_UM_B, scale=scale,
        dy_Bpx=dy_Bpx, dx_Bpx=dx_Bpx, dy_Apx=dy_Apx, dx_Apx=dx_Apx,
        dy_um=dy_um, dx_um=dx_um,
        overlap_peak_px=ov,
        FINAL_MAX_DIST_BPX=FINAL_MAX_DIST_BPX,
        FINAL_MAX_DIST_Apx=FINAL_MAX_DIST_A,
        matches=len(matches), unmatched_A=len(unA), unmatched_B=len(unB),
        A_shape_H=A.shape[0], A_shape_W=A.shape[1],
        B_shape_H=B.shape[0], B_shape_W=B.shape[1],
    )]).to_csv(os.path.join(OUT_DIR, "summary.csv"), index=False)
    print("[Saved] summary.csv")

    # --- Basic plots ---
    if len(df_pairs) >= 2:
        scatter_and_corr(df_pairs["aspectA"],       df_pairs["aspectB"],       "aspect",        OUT_DIR)
        scatter_and_corr(df_pairs["circA"],         df_pairs["circB"],         "circularity",   OUT_DIR)
        scatter_and_corr(df_pairs["solidityA"],     df_pairs["solidityB"],     "solidity",      OUT_DIR)
        scatter_and_corr(df_pairs["eccentricityA"], df_pairs["eccentricityB"], "eccentricity",  OUT_DIR)
        scatter_and_corr(df_pairs["extentA"],       df_pairs["extentB"],       "extent",        OUT_DIR)

    # --- Visualization ---
    bgB = load_bg_image(IMG_B_PATH, (B.shape[0], B.shape[1]), invert=INVERT_VIEW)
    draw_vectors(
        bgB, Axy_B=Axy_B, Bxy_B=Bxy_B, matches=matches,
        upscale=UPSCALE_VIS, out_png=os.path.join(OUT_DIR, "overlay_vectors.png")
    )
    print(f"[Saved] {os.path.join(OUT_DIR, 'overlay_vectors.png')}")


if __name__ == "__main__":
    main()

[SHAPE] A=(1490, 2240), B=(3984, 6000)
[SCALE] 2.6800  A_scaled=(3993, 6003)  B=(3984, 6000)
[COARSE] tried=289  best_ov=1813956  shift=(-20,-20)
[REFINE] tried=3721  best_ov=1883360  shift=(0,-20)
[SELECT] dy_B=0, dx_B=-20, overlap_px=1883360
[MATCH] matches=210  unA=99  unB=44  gate_Apx=37.3
[Saved] pairs.csv
[Saved] summary.csv
[Saved] .\scatter_aspect.png
[Saved] .\scatter_circularity.png
[Saved] .\scatter_solidity.png
[Saved] .\scatter_eccentricity.png


<tifffile.TiffFile 'July_22_ROI_A_004_RGB.tif'> OME series cannot handle discontiguous storage ((3984, 6000, 3) != (3, 3984, 6000))


[Saved] .\scatter_extent.png
[Saved] .\overlay_vectors.png


In [20]:
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from scipy.stats import mannwhitneyu, gaussian_kde

# =========================
# USER SETTINGS
# =========================
PAIR_CSV_DIR = r"C:\Users\oxfil\Pair_for_PCA"
GLOB_PATTERN = "*.csv"
OUT_DIR = r".\shape_pair_pca_analysis"

# Features for drawing
FEATURE_PAIRS = {
    "aspect": ("aspectA", "aspectB"),
    "circularity": ("circA", "circB"),
    "eccentricity": ("eccentricityA", "eccentricityB"),
    "extent": ("extentA", "extentB"),
}

# scaling
SCALER_TYPE = "standard"   # "standard" or "robust"

# random comparison
N_RANDOM_PER_REAL = 20
N_PERMUTATIONS = 2000
RANDOM_SEED = 42

# plotting
FIG_DPI = 300
POINT_SIZE = 14
POINT_ALPHA = 0.28
SAMPLED_PAIR_LINES = 120

USE_DENSITY_CONTOUR = True
CONTOUR_LEVELS = 5
CONTOUR_GRID_N = 200

# optional filtering
MIN_IOU = None      
MAX_DIST_UM = None 

# =========================


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_all_pairs(pair_dir, pattern):
    files = glob(os.path.join(pair_dir, pattern), recursive=True)
    if not files:
        raise FileNotFoundError(f"No pairs.csv found in: {pair_dir}")

    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        df["source_path"] = f
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    return data, files


def filter_pairs(df):
    out = df.copy()

    if MIN_IOU is not None and "iou_Bscale" in out.columns:
        out = out[out["iou_Bscale"] >= MIN_IOU].copy()

    if MAX_DIST_UM is not None and "dist_um" in out.columns:
        out = out[out["dist_um"] <= MAX_DIST_UM].copy()

    return out


def validate_columns(df, feature_pairs):
    missing = []
    for _, (ca, cb) in feature_pairs.items():
        if ca not in df.columns:
            missing.append(ca)
        if cb not in df.columns:
            missing.append(cb)
    if missing:
        raise ValueError(f"Missing columns: {missing}")


def prepare_feature_matrices(df, feature_pairs):
    feature_names = list(feature_pairs.keys())
    colsA = [feature_pairs[f][0] for f in feature_names]
    colsB = [feature_pairs[f][1] for f in feature_names]

    meta_cols = []
    for c in ["idxA", "idxB", "labelA", "labelB", "source_file", "source_path", "dist_um", "iou_Bscale"]:
        if c in df.columns:
            meta_cols.append(c)

    sub = df[colsA + colsB + meta_cols].copy()
    sub = sub.replace([np.inf, -np.inf], np.nan).dropna()

    A = sub[colsA].to_numpy(float)
    B = sub[colsB].to_numpy(float)

    A_df = pd.DataFrame(A, columns=feature_names, index=sub.index)
    B_df = pd.DataFrame(B, columns=feature_names, index=sub.index)

    return sub, A_df, B_df, feature_names


def get_scaler(kind="standard"):
    if kind.lower() == "robust":
        return RobustScaler()
    return StandardScaler()


def joint_normalize(A_df, B_df, scaler_type="standard"):
    scaler = get_scaler(scaler_type)
    pooled = pd.concat([A_df, B_df], axis=0).to_numpy(float)
    scaler.fit(pooled)

    ZA = scaler.transform(A_df.to_numpy(float))
    ZB = scaler.transform(B_df.to_numpy(float))
    return scaler, ZA, ZB


def run_pca(ZA, ZB, n_components=2):
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pooled = np.vstack([ZA, ZB])
    proj = pca.fit_transform(pooled)
    PA = proj[:len(ZA)]
    PB = proj[len(ZA):]
    return pca, PA, PB


def euclidean_pair_distance(A, B):
    return np.sqrt(np.sum((A - B) ** 2, axis=1))


def random_pair_distances(ZA, ZB, n_random_per_real=20, seed=42):
    rng = np.random.default_rng(seed)
    n = len(ZA)
    out = []

    for i in range(n):
        idx = rng.integers(0, n, size=n_random_per_real)
        idx = np.where(idx == i, (idx + 1) % n, idx)
        d = np.sqrt(np.sum((ZA[i][None, :] - ZB[idx]) ** 2, axis=1))
        out.extend(d.tolist())

    return np.asarray(out, dtype=float)


def permutation_test(real_dist, ZA, ZB, n_perm=2000, seed=42):
    rng = np.random.default_rng(seed)
    observed = np.mean(real_dist)

    perm_means = np.zeros(n_perm, dtype=float)
    n = len(ZA)

    for k in range(n_perm):
        perm_idx = rng.permutation(n)
        d = np.sqrt(np.sum((ZA - ZB[perm_idx]) ** 2, axis=1))
        perm_means[k] = np.mean(d)

    p_emp = (np.sum(perm_means <= observed) + 1) / (n_perm + 1)
    return observed, perm_means, p_emp


def add_density_contour(ax, x, y, levels=5, grid_n=200, linewidth=1.2, alpha=0.9):
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) < 20:
        return

    values = np.vstack([x, y])
    kde = gaussian_kde(values)

    xmin, xmax = np.min(x), np.max(x)
    ymin, ymax = np.min(y), np.max(y)

    xpad = 0.08 * (xmax - xmin + 1e-12)
    ypad = 0.08 * (ymax - ymin + 1e-12)

    xx, yy = np.meshgrid(
        np.linspace(xmin - xpad, xmax + xpad, grid_n),
        np.linspace(ymin - ypad, ymax + ypad, grid_n)
    )
    coords = np.vstack([xx.ravel(), yy.ravel()])
    zz = kde(coords).reshape(xx.shape)

    zmin, zmax = np.nanmin(zz), np.nanmax(zz)
    levs = np.linspace(zmin + 0.2 * (zmax - zmin), zmax, levels)
    ax.contour(xx, yy, zz, levels=levs, linewidths=linewidth, alpha=alpha)


def plot_pca_overlay(PA, PB, explained, out_path):
    fig, ax = plt.subplots(figsize=(6.2, 5.4))

    ax.scatter(PA[:, 0], PA[:, 1], s=POINT_SIZE, alpha=POINT_ALPHA, label="Method A")
    ax.scatter(PB[:, 0], PB[:, 1], s=POINT_SIZE, alpha=POINT_ALPHA, label="Method B")

    if USE_DENSITY_CONTOUR:
        add_density_contour(ax, PA[:, 0], PA[:, 1], levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N)
        add_density_contour(ax, PB[:, 0], PB[:, 1], levels=CONTOUR_LEVELS, grid_n=CONTOUR_GRID_N)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def plot_sampled_pair_lines(PA, PB, explained, out_path, sample_n=120, seed=42):
    rng = np.random.default_rng(seed)
    n = len(PA)
    sample_n = min(sample_n, n)
    idx = rng.choice(n, size=sample_n, replace=False)

    fig, ax = plt.subplots(figsize=(6.2, 5.4))
    ax.scatter(PA[:, 0], PA[:, 1], s=8, alpha=0.16, label="Method A")
    ax.scatter(PB[:, 0], PB[:, 1], s=8, alpha=0.16, label="Method B")

    for i in idx:
        ax.plot([PA[i, 0], PB[i, 0]], [PA[i, 1], PB[i, 1]], lw=0.7, alpha=0.6)

    ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def plot_real_vs_random(real_dist, rand_dist, p_mwu, p_emp, out_path):
    fig, ax = plt.subplots(figsize=(5.8, 5.0))

    data = [real_dist, rand_dist]
    labels = ["Real pair", "Random pair"]

    vp = ax.violinplot(data, showmeans=False, showmedians=True, showextrema=False)
    for body in vp["bodies"]:
        body.set_alpha(0.45)

    rng = np.random.default_rng(RANDOM_SEED)
    for i, arr in enumerate(data, start=1):
        x = rng.normal(i, 0.05, size=len(arr))
        ax.scatter(x, arr, s=6, alpha=0.08)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(labels)
    ax.set_ylabel("Distance in normalized shape space")
    ax.set_title(
        f"MWU p = {p_mwu:.3e}\nPermutation p = {p_emp:.3e}\n"
        f"Median(real) = {np.median(real_dist):.3f}, Median(random) = {np.median(rand_dist):.3f}",
        fontsize=10
    )

    fig.tight_layout()
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def plot_loadings(pca, feature_names, out_png, out_csv):
    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_names,
        columns=[f"PC{i+1}" for i in range(pca.components_.shape[0])]
    )
    loadings.to_csv(out_csv)

    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    x = np.arange(len(feature_names))
    ax.bar(x - 0.18, loadings["PC1"], width=0.36, label="PC1")
    ax.bar(x + 0.18, loadings["PC2"], width=0.36, label="PC2")
    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=30, ha="right")
    ax.set_ylabel("Loading")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)


def save_pair_level(sub, PA, PB, real_dist, out_csv):
    out = sub.copy()
    out["A_PC1"] = PA[:, 0]
    out["A_PC2"] = PA[:, 1]
    out["B_PC1"] = PB[:, 0]
    out["B_PC2"] = PB[:, 1]
    out["shape_pair_distance"] = real_dist
    out.to_csv(out_csv, index=False)


def save_summary(real_dist, rand_dist, pca, p_mwu, p_emp, out_csv):
    summary = pd.DataFrame([{
        "n_real_pairs": len(real_dist),
        "n_random_pairs": len(rand_dist),
        "real_mean": float(np.mean(real_dist)),
        "real_median": float(np.median(real_dist)),
        "random_mean": float(np.mean(rand_dist)),
        "random_median": float(np.median(rand_dist)),
        "mannwhitney_p": float(p_mwu),
        "permutation_p": float(p_emp),
        "pc1_var_ratio": float(pca.explained_variance_ratio_[0]),
        "pc2_var_ratio": float(pca.explained_variance_ratio_[1]),
    }])
    summary.to_csv(out_csv, index=False)


def main():
    ensure_dir(OUT_DIR)

    df, files = load_all_pairs(PAIR_CSV_DIR, GLOB_PATTERN)
    print(f"Loaded files: {len(files)}")
    print(f"Rows before filtering: {len(df)}")

    validate_columns(df, FEATURE_PAIRS)
    df = filter_pairs(df)
    print(f"Rows after filtering: {len(df)}")

    sub, A_df, B_df, feature_names = prepare_feature_matrices(df, FEATURE_PAIRS)
    print(f"Valid rows for PCA/similarity: {len(sub)}")

    scaler, ZA, ZB = joint_normalize(A_df, B_df, scaler_type=SCALER_TYPE)
    pca, PA, PB = run_pca(ZA, ZB, n_components=2)

    real_dist = euclidean_pair_distance(ZA, ZB)
    rand_dist = random_pair_distances(
        ZA, ZB,
        n_random_per_real=N_RANDOM_PER_REAL,
        seed=RANDOM_SEED
    )

    _, p_mwu = mannwhitneyu(real_dist, rand_dist, alternative="less")
    _, perm_means, p_emp = permutation_test(
        real_dist, ZA, ZB,
        n_perm=N_PERMUTATIONS,
        seed=RANDOM_SEED
    )

    save_pair_level(
        sub, PA, PB, real_dist,
        os.path.join(OUT_DIR, "pair_level_shape_similarity.csv")
    )
    save_summary(
        real_dist, rand_dist, pca, p_mwu, p_emp,
        os.path.join(OUT_DIR, "summary_shape_similarity.csv")
    )

    plot_pca_overlay(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_overlay_A_vs_B.png")
    )
    plot_sampled_pair_lines(
        PA, PB, pca.explained_variance_ratio_,
        os.path.join(OUT_DIR, "pca_pairs_connected.png"),
        sample_n=SAMPLED_PAIR_LINES,
        seed=RANDOM_SEED
    )
    plot_real_vs_random(
        real_dist, rand_dist, p_mwu, p_emp,
        os.path.join(OUT_DIR, "real_vs_random_pair_distance.png")
    )
    plot_loadings(
        pca, feature_names,
        os.path.join(OUT_DIR, "pca_loadings.png"),
        os.path.join(OUT_DIR, "pca_loadings.csv")
    )

    print("Saved:")
    print(" - pair_level_shape_similarity.csv")
    print(" - summary_shape_similarity.csv")
    print(" - pca_overlay_A_vs_B.png")
    print(" - pca_pairs_connected.png")
    print(" - real_vs_random_pair_distance.png")
    print(" - pca_loadings.png")
    print(" - pca_loadings.csv")


if __name__ == "__main__":
    main()

Loaded files: 6
Rows before filtering: 1071
Rows after filtering: 1071
Valid rows for PCA/similarity: 1071
Saved:
 - pair_level_shape_similarity.csv
 - summary_shape_similarity.csv
 - pca_overlay_A_vs_B.png
 - pca_pairs_connected.png
 - real_vs_random_pair_distance.png
 - pca_loadings.png
 - pca_loadings.csv


In [ ]:
import os
import glob
import itertools
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde
from skimage.measure import regionprops_table, label
from skimage.io import imread

matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['savefig.transparent'] = True

save_fig = True

condition_folders = {
    "Capacitance": {
        "folder": r"/Volumes/ENG_BBCancer_Shared/group/Tae/Analysis_for_size_260420/CMOS_threshold",
        "type": "binary_image"
    },
    "Fluorescence": {
        "folder": r"/Volumes/ENG_BBCancer_Shared/group/Tae/Analysis_for_size_260420/Optical",
        "type": "npy"
    },
    "Phase_control": {
        "folder": r"/Volumes/ENG_BBCancer_Shared/group/Tae/Analysis_for_size_260420/Phase",
        "type": "npy"
    },
}

condition_scales = {
    "Capacitance": 10.00,
    "Fluorescence": 0.373,
    "Phase_control": 1.27,
}


binary_foreground_white = {
    "Capacitance": False,
}

binary_image_extensions = ("*.tif", "*.tiff", "*.png", "*.jpg", "*.jpeg", "*.bmp")

condition_colors = {
    "Capacitance": "#1f77b4",   # blue
    "Fluorescence": "#2ca02c",  # red
    "Phase_control": "#d62728", # green
}


rounding_conditions = {"Fluorescence", "Phase_control"}
rounding_unit_um = 1.0

rounding_mode = "round"

# Feature selection
# "major_axis_um"
# "minor_axis_um"
# "major_minor_product_um2"
# "ellipse_area_um2"
selected_feature = "major_axis_um"

feature_configs = {
    "major_axis_um": {
        "xlabel": "Major axis ($\\mu m$)",
        "title": "Major-axis distribution by condition",
        "x_range": (0, 250),
        "bin_width": 5.0,
        "output_plot": "condition_comparison_major_axis_rounded_KDE_hist.pdf",
        "output_csv": "all_conditions_major_axis_rounded_measurements.csv",
        "output_ovl_csv": "condition_major_axis_rounded_overlap_matrix.csv",
        "output_pair_csv": "condition_major_axis_rounded_pairwise_overlap.csv",
    },
    "minor_axis_um": {
        "xlabel": "Minor axis ($\\mu m$)",
        "title": "Minor-axis distribution by condition",
        "x_range": (0, 100),
        "bin_width": 5.0,
        "output_plot": "condition_comparison_minor_axis_rounded_KDE_hist.pdf",
        "output_csv": "all_conditions_minor_axis_rounded_measurements.csv",
        "output_ovl_csv": "condition_minor_axis_rounded_overlap_matrix.csv",
        "output_pair_csv": "condition_minor_axis_rounded_pairwise_overlap.csv",
    },
    "major_minor_product_um2": {
        "xlabel": "Major × Minor ($\\mu m^2$)",
        "title": "Major×Minor distribution by condition",
        "x_range": (0, 12000),
        "bin_width": 50,
        "output_plot": "condition_comparison_major_minor_product_rounded_KDE_hist.pdf",
        "output_csv": "all_conditions_major_minor_product_rounded_measurements.csv",
        "output_ovl_csv": "condition_major_minor_product_rounded_overlap_matrix.csv",
        "output_pair_csv": "condition_major_minor_product_rounded_pairwise_overlap.csv",
    },
    "ellipse_area_um2": {
        "xlabel": "Ellipse area from major/minor ($\\mu m^2$)",
        "title": "Rounded ellipse-area distribution by condition",
        "x_range": (0, 10000),
        "bin_width": 50,
        "output_plot": "condition_comparison_ellipse_area_rounded_KDE_hist.pdf",
        "output_csv": "all_conditions_ellipse_area_rounded_measurements.csv",
        "output_ovl_csv": "condition_ellipse_area_rounded_overlap_matrix.csv",
        "output_pair_csv": "condition_ellipse_area_rounded_pairwise_overlap.csv",
    },
}

cfg = feature_configs[selected_feature]
x_range = cfg["x_range"]
bin_width = cfg["bin_width"]
bin_edges = np.arange(x_range[0], x_range[1] + bin_width, bin_width)

y_range = None
n_points = 1000

feature_min = 0
feature_max = None


def round_to_unit(values, unit=10.0, mode="round"):
    values = np.asarray(values, dtype=float)

    if mode == "round":
        return np.round(values / unit) * unit
    elif mode == "ceil":
        return np.ceil(values / unit) * unit
    elif mode == "floor":
        return np.floor(values / unit) * unit
    elif mode == "none":
        return values
    else:
        raise ValueError(f"Unknown rounding mode: {mode}")


def load_cellpose_mask(npy_path):
    data = np.load(npy_path, allow_pickle=True)

    if isinstance(data, np.ndarray) and data.shape == () and data.dtype == object:
        data = data.item()

    if isinstance(data, dict):
        for key in ["masks", "mask", "cell_mask"]:
            if key in data:
                return data[key]
        raise KeyError(f"No valid mask key found in: {npy_path}")

    return data


def load_binary_mask_and_label(img_path, foreground_white=True):
    img = imread(img_path)

    if img.ndim == 3:
        img = img[..., 0]

    if foreground_white:
        binary = img > 0
    else:
        binary = img == 0

    if np.sum(binary) == 0:
        return np.zeros_like(binary, dtype=np.int32)

    labeled = label(binary, connectivity=2)
    return labeled.astype(np.int32)


def extract_axis_features_from_mask(mask, pixel_size_um):
    props = regionprops_table(
        mask,
        properties=["label", "area", "major_axis_length", "minor_axis_length"]
    )
    df = pd.DataFrame(props)

    if df.empty:
        return df

    df["area_px"] = df["area"].astype(float)
    df["major_axis_px"] = df["major_axis_length"].astype(float)
    df["minor_axis_px"] = df["minor_axis_length"].astype(float)

    # 원본 um
    df["major_axis_um_raw"] = df["major_axis_px"] * pixel_size_um
    df["minor_axis_um_raw"] = df["minor_axis_px"] * pixel_size_um
    df["area_um2_raw"] = df["area_px"] * (pixel_size_um ** 2)

    return df


def apply_condition_specific_rounding(df, condition):
    df = df.copy()

    if condition in rounding_conditions:
        df["major_axis_um"] = round_to_unit(
            df["major_axis_um_raw"].values,
            unit=rounding_unit_um,
            mode=rounding_mode
        )
        df["minor_axis_um"] = round_to_unit(
            df["minor_axis_um_raw"].values,
            unit=rounding_unit_um,
            mode=rounding_mode
        )
    else:
        df["major_axis_um"] = df["major_axis_um_raw"]
        df["minor_axis_um"] = df["minor_axis_um_raw"]
    df["major_minor_product_um2"] = df["major_axis_um"] * df["minor_axis_um"]
    df["ellipse_area_um2"] = (np.pi / 4.0) * df["major_axis_um"] * df["minor_axis_um"]

    return df


def compute_ovl_from_kde(values_a, values_b, x_grid):
    if len(values_a) < 4 or len(values_b) < 4:
        return np.nan, None, None

    if np.std(values_a) == 0 or np.std(values_b) == 0:
        return np.nan, None, None

    kde_a = gaussian_kde(values_a)
    kde_b = gaussian_kde(values_b)

    y_a = kde_a(x_grid)
    y_b = kde_b(x_grid)

    ovl = np.trapz(np.minimum(y_a, y_b), x_grid)
    return ovl, y_a, y_b


def collect_files(folder, data_type):
    if data_type == "npy":
        return sorted(glob.glob(os.path.join(folder, "*.npy")))
    elif data_type == "binary_image":
        files = []
        for ext in binary_image_extensions:
            files.extend(glob.glob(os.path.join(folder, ext)))
        return sorted(files)
    else:
        raise ValueError(f"Unknown data type: {data_type}")


# =========================
# Data Load
# =========================
all_dfs = []

for condition, info in condition_folders.items():
    folder = info["folder"]
    data_type = info["type"]
    pixel_size_um = condition_scales[condition]

    files = collect_files(folder, data_type)

    if len(files) == 0:
        print(f"[Warning] No files found in {condition}: {folder}")
        continue

    cond_dfs = []

    for f in files:
        try:
            if data_type == "npy":
                mask = load_cellpose_mask(f)
                if mask.ndim != 2:
                    print(f"[Skip] {condition} | {os.path.basename(f)} shape={mask.shape}")
                    continue
            elif data_type == "binary_image":
                fg_white = binary_foreground_white.get(condition, True)
                mask = load_binary_mask_and_label(f, foreground_white=fg_white)

            df = extract_axis_features_from_mask(mask, pixel_size_um)

            if not df.empty:
                df = apply_condition_specific_rounding(df, condition)
                df["condition"] = condition
                df["source_file"] = os.path.basename(f)
                df["input_type"] = data_type
                cond_dfs.append(df)

        except Exception as e:
            print(f"[Failed] {condition} | {os.path.basename(f)} -> {e}")

    if cond_dfs:
        all_dfs.append(pd.concat(cond_dfs, ignore_index=True))

if not all_dfs:
    raise ValueError("No valid data extracted from any condition.")

result_df = pd.concat(all_dfs, ignore_index=True)

if feature_min is not None:
    result_df = result_df[result_df[selected_feature] >= feature_min]
if feature_max is not None:
    result_df = result_df[result_df[selected_feature] <= feature_max]

if len(result_df) == 0:
    raise ValueError(f"No valid values remain after filtering for {selected_feature}.")

result_df.to_csv(cfg["output_csv"], index=False)
print(f"Saved per-cell data: {cfg['output_csv']}")


x_grid = np.linspace(x_range[0], x_range[1], n_points)

# =========================
# KDE + Histogram plot
# =========================
plt.figure(figsize=(8, 6))

condition_values = {}
for condition in condition_folders.keys():
    vals = result_df.loc[result_df["condition"] == condition, selected_feature].dropna().values
    condition_values[condition] = vals

    if len(vals) == 0:
        continue

    color = condition_colors.get(condition, None)

    plt.hist(
        vals,
        bins=bin_edges,
        density=True,
        alpha=0.22,
        color=color,
        edgecolor=color,
        linewidth=1.0,
        histtype='stepfilled',
        label=None
    )

    if len(vals) >= 4 and np.std(vals) > 0:
        kde = gaussian_kde(vals)
        y = kde(x_grid)
        plt.plot(
            x_grid, y,
            color=color,
            linewidth=2.5,
            label=f"{condition} (n={len(vals)})"
        )
    else:
        print(f"[Skip KDE] {condition}")

plt.xlabel(cfg["xlabel"])
plt.ylabel("Density")
plt.title(cfg["title"])
plt.xlim(x_range)

if y_range is not None:
    plt.ylim(y_range)

plt.legend(frameon=False)
plt.tight_layout()

if save_fig:
    plt.savefig(cfg["output_plot"], dpi=300, bbox_inches="tight", transparent=True)
    print(f"Saved figure: {cfg['output_plot']}")

plt.show()

# =========================
# Pairwise OVL
# =========================
conditions = list(condition_folders.keys())

pair_rows = []
ovl_matrix = pd.DataFrame(np.nan, index=conditions, columns=conditions)

for c in conditions:
    ovl_matrix.loc[c, c] = 1.0

for c1, c2 in itertools.combinations(conditions, 2):
    v1 = condition_values.get(c1, np.array([]))
    v2 = condition_values.get(c2, np.array([]))

    ovl, _, _ = compute_ovl_from_kde(v1, v2, x_grid)

    pair_rows.append({
        "condition_1": c1,
        "condition_2": c2,
        "overlap_ratio": ovl
    })

    ovl_matrix.loc[c1, c2] = ovl
    ovl_matrix.loc[c2, c1] = ovl

pair_df = pd.DataFrame(pair_rows)
pair_df.to_csv(cfg["output_pair_csv"], index=False)
ovl_matrix.to_csv(cfg["output_ovl_csv"])

print("\nPairwise overlap ratios:")
print(pair_df)

print("\nOverlap matrix:")
print(ovl_matrix)

In [ ]:
# For statistics
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools
import numpy as np
import pandas as pd


# =========================
# Statistics settings
# =========================
ALPHA = 0.05

# For hypothesis tests, use sample/image-level summary.
# Recommended:
#   "median" = robust against skewed cell-size distributions
#   "mean"   = tests average morphology more directly
sample_summary_metric = "median"   # "median" or "mean"

# If True, run statistics on sample-level values.
# Keep this True for publication-level statistics.
use_sample_level_statistics = True


# =========================
# Output filenames
# =========================
base_name = cfg["output_csv"].replace(".csv", "")

spss_raw_csv = f"{base_name}_SPSS_raw_cell_level.csv"
spss_sample_csv = f"{base_name}_SPSS_sample_level_summary.csv"

normality_csv = f"{base_name}_sample_level_normality.csv"
omnibus_csv = f"{base_name}_sample_level_omnibus_test.csv"
pairwise_csv = f"{base_name}_sample_level_pairwise_posthoc.csv"
descriptive_csv = f"{base_name}_sample_level_descriptive_stats.csv"


# =========================
# 1. Export raw cell-level data for SPSS
# =========================
spss_raw_df = result_df.copy()

spss_raw_df["selected_feature"] = selected_feature
spss_raw_df["value_for_analysis"] = spss_raw_df[selected_feature]

# Keep useful columns first
front_cols = [
    "condition",
    "source_file",
    "label",
    "selected_feature",
    "value_for_analysis",
    "major_axis_um_raw",
    "minor_axis_um_raw",
    "area_um2_raw",
    "major_axis_um",
    "minor_axis_um",
    "major_minor_product_um2",
    "ellipse_area_um2",
    "input_type",
    "pixel_size_um",
]

existing_front_cols = [c for c in front_cols if c in spss_raw_df.columns]
remaining_cols = [c for c in spss_raw_df.columns if c not in existing_front_cols]

spss_raw_df = spss_raw_df[existing_front_cols + remaining_cols]
spss_raw_df.to_csv(spss_raw_csv, index=False)

print(f"\nSaved SPSS raw cell-level CSV:")
print(spss_raw_csv)


# =========================
# 2. Create sample-level summary
# =========================
sample_df = (
    result_df
    .groupby(["condition", "source_file"], as_index=False)
    .agg(
        n_cells=(selected_feature, "count"),
        sample_mean=(selected_feature, "mean"),
        sample_median=(selected_feature, "median"),
        sample_std=(selected_feature, "std"),
        sample_q25=(selected_feature, lambda x: np.nanpercentile(x, 25)),
        sample_q75=(selected_feature, lambda x: np.nanpercentile(x, 75)),
    )
)

sample_df["sample_iqr"] = sample_df["sample_q75"] - sample_df["sample_q25"]

if sample_summary_metric == "mean":
    sample_df["value_for_analysis"] = sample_df["sample_mean"]
elif sample_summary_metric == "median":
    sample_df["value_for_analysis"] = sample_df["sample_median"]
else:
    raise ValueError("sample_summary_metric must be 'mean' or 'median'.")

sample_df["selected_feature"] = selected_feature
sample_df["summary_metric_used"] = sample_summary_metric

sample_df.to_csv(spss_sample_csv, index=False)

print(f"\nSaved SPSS sample-level summary CSV:")
print(spss_sample_csv)


# =========================
# 3. Descriptive statistics by condition
# =========================
descriptive_rows = []

for condition, sub in sample_df.groupby("condition"):
    vals = sub["value_for_analysis"].dropna().values.astype(float)

    descriptive_rows.append({
        "condition": condition,
        "selected_feature": selected_feature,
        "summary_metric_used": sample_summary_metric,
        "n_samples": len(vals),
        "mean_of_sample_values": np.nanmean(vals) if len(vals) > 0 else np.nan,
        "median_of_sample_values": np.nanmedian(vals) if len(vals) > 0 else np.nan,
        "std_of_sample_values": np.nanstd(vals, ddof=1) if len(vals) > 1 else np.nan,
        "sem_of_sample_values": (
            np.nanstd(vals, ddof=1) / np.sqrt(len(vals))
            if len(vals) > 1 else np.nan
        ),
        "q25_of_sample_values": np.nanpercentile(vals, 25) if len(vals) > 0 else np.nan,
        "q75_of_sample_values": np.nanpercentile(vals, 75) if len(vals) > 0 else np.nan,
    })

descriptive_df = pd.DataFrame(descriptive_rows)
descriptive_df.to_csv(descriptive_csv, index=False)

print("\nSample-level descriptive statistics:")
print(descriptive_df)
print(f"\nSaved descriptive stats:")
print(descriptive_csv)


# =========================
# 4. Normality test by condition
# =========================
normality_rows = []
condition_sample_values = {}

for condition in condition_folders.keys():
    vals = sample_df.loc[
        sample_df["condition"] == condition,
        "value_for_analysis"
    ].dropna().values.astype(float)

    vals = vals[np.isfinite(vals)]
    condition_sample_values[condition] = vals

    if len(vals) < 3:
        normality_rows.append({
            "condition": condition,
            "selected_feature": selected_feature,
            "summary_metric_used": sample_summary_metric,
            "n_samples": len(vals),
            "normality_test": "Shapiro-Wilk",
            "W_statistic": np.nan,
            "p_value": np.nan,
            "normality_pass": False,
            "note": "n < 3; Shapiro-Wilk cannot be applied"
        })
        continue

    if np.nanstd(vals, ddof=1) == 0:
        normality_rows.append({
            "condition": condition,
            "selected_feature": selected_feature,
            "summary_metric_used": sample_summary_metric,
            "n_samples": len(vals),
            "normality_test": "Shapiro-Wilk",
            "W_statistic": np.nan,
            "p_value": np.nan,
            "normality_pass": False,
            "note": "zero variance; normality cannot be meaningfully tested"
        })
        continue

    shapiro_res = stats.shapiro(vals)

    normality_rows.append({
        "condition": condition,
        "selected_feature": selected_feature,
        "summary_metric_used": sample_summary_metric,
        "n_samples": len(vals),
        "normality_test": "Shapiro-Wilk",
        "W_statistic": float(shapiro_res.statistic),
        "p_value": float(shapiro_res.pvalue),
        "normality_pass": bool(shapiro_res.pvalue >= ALPHA),
        "note": ""
    })

normality_df = pd.DataFrame(normality_rows)
normality_df.to_csv(normality_csv, index=False)

print("\nNormality test by condition:")
print(normality_df)
print(f"\nSaved normality results:")
print(normality_csv)


# =========================
# 5. Welch ANOVA function
# =========================
def welch_anova(*groups):
    """
    Welch ANOVA for independent groups with unequal variance.
    Returns F statistic, df1, df2, p value.
    """

    groups = [
        np.asarray(g, dtype=float)
        for g in groups
    ]

    groups = [
        g[np.isfinite(g)]
        for g in groups
    ]

    groups = [
        g for g in groups
        if len(g) >= 2 and np.nanstd(g, ddof=1) > 0
    ]

    k = len(groups)

    if k < 2:
        return np.nan, np.nan, np.nan, np.nan

    n = np.array([len(g) for g in groups], dtype=float)
    means = np.array([np.nanmean(g) for g in groups], dtype=float)
    variances = np.array([np.nanvar(g, ddof=1) for g in groups], dtype=float)

    weights = n / variances
    weighted_mean = np.sum(weights * means) / np.sum(weights)

    numerator = np.sum(weights * (means - weighted_mean) ** 2) / (k - 1)

    correction = np.sum(
        ((1 - weights / np.sum(weights)) ** 2) / (n - 1)
    )

    denominator = 1 + (2 * (k - 2) / (k**2 - 1)) * correction

    F_stat = numerator / denominator

    df1 = k - 1
    df2 = (k**2 - 1) / (3 * correction)

    p_value = stats.f.sf(F_stat, df1, df2)

    return float(F_stat), float(df1), float(df2), float(p_value)


# =========================
# 6. Select omnibus test
# =========================
valid_conditions = [
    c for c, vals in condition_sample_values.items()
    if len(vals) >= 2 and np.nanstd(vals, ddof=1) > 0
]

valid_groups = [
    condition_sample_values[c]
    for c in valid_conditions
]

# Strict rule:
# all valid conditions must pass normality
normality_valid_df = normality_df[
    normality_df["condition"].isin(valid_conditions)
].copy()

all_normal = (
    len(valid_conditions) >= 2
    and len(normality_valid_df) == len(valid_conditions)
    and normality_valid_df["normality_pass"].all()
)

if all_normal:
    omnibus_test = "Welch ANOVA"

    F_stat, df1, df2, p_value = welch_anova(*valid_groups)

    omnibus_df = pd.DataFrame([{
        "selected_feature": selected_feature,
        "summary_metric_used": sample_summary_metric,
        "test_selected": omnibus_test,
        "selection_rule": "All valid conditions passed Shapiro-Wilk normality test",
        "conditions_included": "; ".join(valid_conditions),
        "n_conditions": len(valid_conditions),
        "statistic_name": "F",
        "statistic": F_stat,
        "df1": df1,
        "df2": df2,
        "p_value": p_value,
        "alpha": ALPHA,
        "significant": bool(p_value < ALPHA) if np.isfinite(p_value) else False,
    }])

else:
    omnibus_test = "Kruskal-Wallis"

    if len(valid_groups) >= 2:
        kw_res = stats.kruskal(*valid_groups)
        statistic = float(kw_res.statistic)
        p_value = float(kw_res.pvalue)
    else:
        statistic = np.nan
        p_value = np.nan

    omnibus_df = pd.DataFrame([{
        "selected_feature": selected_feature,
        "summary_metric_used": sample_summary_metric,
        "test_selected": omnibus_test,
        "selection_rule": "At least one valid condition failed or could not complete Shapiro-Wilk normality test",
        "conditions_included": "; ".join(valid_conditions),
        "n_conditions": len(valid_conditions),
        "statistic_name": "H",
        "statistic": statistic,
        "df1": len(valid_conditions) - 1 if len(valid_conditions) >= 2 else np.nan,
        "df2": np.nan,
        "p_value": p_value,
        "alpha": ALPHA,
        "significant": bool(p_value < ALPHA) if np.isfinite(p_value) else False,
    }])

omnibus_df.to_csv(omnibus_csv, index=False)

print("\nSelected omnibus test:")
print(omnibus_df)
print(f"\nSaved omnibus result:")
print(omnibus_csv)


# =========================
# 7. Pairwise post-hoc tests
# =========================
pairwise_rows = []

for c1, c2 in itertools.combinations(valid_conditions, 2):
    v1 = condition_sample_values[c1]
    v2 = condition_sample_values[c2]

    v1 = v1[np.isfinite(v1)]
    v2 = v2[np.isfinite(v2)]

    if len(v1) < 2 or len(v2) < 2:
        continue

    if omnibus_test == "Welch ANOVA":
        pairwise_test = "Welch t-test"
        stat_name = "t"

        res = stats.ttest_ind(
            v1,
            v2,
            equal_var=False,
            nan_policy="omit"
        )

        stat_value = float(res.statistic)
        raw_p = float(res.pvalue)

    else:
        pairwise_test = "Mann-Whitney U test"
        stat_name = "U"

        res = stats.mannwhitneyu(
            v1,
            v2,
            alternative="two-sided"
        )

        stat_value = float(res.statistic)
        raw_p = float(res.pvalue)

    pairwise_rows.append({
        "selected_feature": selected_feature,
        "summary_metric_used": sample_summary_metric,
        "condition_1": c1,
        "condition_2": c2,
        "pairwise_test": pairwise_test,
        "statistic_name": stat_name,
        "statistic": stat_value,
        "raw_p_value": raw_p,

        "n_samples_1": len(v1),
        "n_samples_2": len(v2),

        "mean_1": np.nanmean(v1),
        "mean_2": np.nanmean(v2),
        "median_1": np.nanmedian(v1),
        "median_2": np.nanmedian(v2),
        "std_1": np.nanstd(v1, ddof=1),
        "std_2": np.nanstd(v2, ddof=1),

        "mean_difference_2_minus_1": np.nanmean(v2) - np.nanmean(v1),
        "median_difference_2_minus_1": np.nanmedian(v2) - np.nanmedian(v1),
    })

pairwise_df = pd.DataFrame(pairwise_rows)

if len(pairwise_df) > 0:
    reject, p_holm, _, _ = multipletests(
        pairwise_df["raw_p_value"].values,
        alpha=ALPHA,
        method="holm"
    )

    pairwise_df["holm_adjusted_p_value"] = p_holm
    pairwise_df["significant_after_holm_alpha_0_05"] = reject

pairwise_df.to_csv(pairwise_csv, index=False)

print("\nPairwise post-hoc results:")
print(pairwise_df)
print(f"\nSaved pairwise post-hoc results:")
print(pairwise_csv)

Figure subset of E,F

In [19]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 28
mpl.rcParams["axes.linewidth"] = 1.2
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42


def plot_boxplot_from_sampled_csv(raw_csv_path, save_path=None):
    df = pd.read_csv(raw_csv_path)

    required_cols = {"region", "intensity"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required_cols}")

    cell_vals = df.loc[df["region"] == "Cell", "intensity"].dropna().values
    bg_vals = df.loc[df["region"] == "Background", "intensity"].dropna().values

    print(f"Cell pixels used:       {cell_vals.size}")
    print(f"Background pixels used: {bg_vals.size}")

    if cell_vals.size == 0 or bg_vals.size == 0:
        raise ValueError("Cell or Background data is empty.")

    fig, ax = plt.subplots(figsize=(3.2, 3.2))

    bp = ax.boxplot(
        [cell_vals, bg_vals],
        labels=["Cell", "Background"],
        showfliers=False,
        widths=0.6
    )

    for element in ["boxes", "whiskers", "caps", "medians"]:
        for line in bp[element]:
            line.set_linewidth(1.2)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="in", length=4, width=1.0)

    fig.tight_layout()

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.patch.set_alpha(0)
        ax.set_facecolor("none")
        plt.savefig(save_path, bbox_inches="tight", transparent=True)
        plt.close()
        print(f"Saved boxplot: {save_path}")
    else:
        plt.show()


raw_csv = r"C:\Users\oxfil\Cell_optical_media_comparision_new\inside_vs_outside_raw_pixel_intensity_cmos.csv"

out_pdf = r"C:\Users\oxfil\Cell_optical_media_comparision_new\inside_vs_outside_boxplot_pixel_intensity_cmos.pdf"

plot_boxplot_from_sampled_csv(
    raw_csv_path=raw_csv,
    save_path=out_pdf
)

Cell pixels used:       1662549
Background pixels used: 8208851


C:\Users\oxfil\AppData\Local\Temp\ipykernel_15236\3479521449.py:32: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


Saved boxplot: C:\Users\oxfil\Cell_optical_media_comparision_new\inside_vs_outside_boxplot_pixel_intensity_cmos.pdf
